<a href="https://colab.research.google.com/github/joanby/tensorflow2/blob/master/Collab%208%20-%20Reinforcement%20Learning%20para%20problemas%20de%20Stock%20Market%20Trading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Paso 1: Instalar las dependencias y configurar el entorno

In [8]:
!pip install -U "tensorflow==2.20.0"

In [9]:
!pip install -U yfinance

In [10]:
!pip install pandas-datareader

## Paso 2: Importar las dependencias del proyecto

In [11]:
import yfinance as yf
import math
import random
import numpy as np
import pandas as pd
import tensorflow as tf

import matplotlib.pyplot as plt
import pandas_datareader as data_reader

from tqdm import tqdm_notebook, tqdm
from collections import deque

In [12]:
tf.__version__

'2.20.0'

## Paso 3: Construir la red neuronal de la AI del Trader

In [13]:
# Esta clase sera un boot
class AI_Trader():
  # Creamos el constructor con el nombre del modelo como AITrader
  def __init__(self, state_size, action_space=3, model_name="AITrader"): #Manten, Compra, Vende

    # Tamaño de los estados
    self.state_size = state_size
    # Numero de accion que va a poder tomar
    self.action_space = action_space
    # definimos la momoria de longitud maxima deque 2.000
    self.memory = deque(maxlen=2000)
    # Inventario es una lista vacia que guardaremos las divisas
    self.inventory = []
    self.model_name = model_name

    # Definimos para la fase de aprendizaje:
    self.gamma = 0.95
    self.epsilon = 1.0
    self.epsilon_final = 0.01
    self.epsilon_decay = 0.995

    # Inicializamos la funcion que crearemos a continuacion
    self.model = self.model_builder()

  # Esta sera la funcion de crear el modelo
  def model_builder(self):
    # Definimos el modelo a trabajar utilizando la clase Sequential
    model = tf.keras.models.Sequential()

    # Ahora la capa de entrada indicando el numero de estados que tendra
    model.add(tf.keras.layers.Dense(units=32, activation='relu', input_dim=self.state_size))

    # Creamos mas capas ocultas duplicando las neuronas
    model.add(tf.keras.layers.Dense(units=64, activation='relu'))
    model.add(tf.keras.layers.Dense(units=128, activation='relu'))

    # Esta es la capa de salida con el espacio de salida con la funcion de activacion lineal
    model.add(tf.keras.layers.Dense(units=self.action_space, activation='linear'))

    # Ahora compilamos el modelo, con la funcion de perdida media, con el optimizador de adam
    model.compile(loss='mse', optimizer=tf.keras.optimizers.Adam(lr=0.001))

    # Devolvemos el modelo
    return model

  # Funcion de trade para la toma de deciciones de comprar, vender o mantener
  def trade(self, state):
    # SI al generar un numero aleatorio, el numero sale mas pequeño que el
    if random.random() <= self.epsilon:
      # devolvemos un numero aleatorio entre 0 y 2 incluyendo ambos
      return random.randrange(self.action_space)

    # En caso contrario definimos el confunto de acciones de acuerdo a la prediccion de acuerdo al estado actual
    actions = self.model.predict(state)
    # Devolvemos la accion con mayor probabilidad
    return np.argmax(actions[0])

  # Funcion para entrenar con el argumento del tamaño del bloque a entrenar
  def batch_train(self, batch_size):

    # Creamos una lista vacia
    batch = []
    # Iteramos dentro del bloque en memoria - batch_size de la longitud de la memoria (eteramos de atras hacia adelante)
    for i in range(len(self.memory) - batch_size + 1, len(self.memory)):
      # Apendizamos los ultimos bloques de memoria
      batch.append(self.memory[i])

    # Cada uno de los bloques que hay en la memoria contara con un estado, una accion, la recompenza, el estado siguiente y saber si finalizo el ejercicio
    for state, action, reward, next_state, done in batch:
      # definimos la variable reward
      reward = reward
      # Si no hemos terminado
      if not done:
        # actualizamos la recompenza
        reward = reward + self.gamma * np.amax(self.model.predict(next_state)[0])

      # El targe sera el que me predice el modelo a partir de un estado
      target = self.model.predict(state)
      # Para la accion actual que toma el agente, sera la recompenza que hemos calculado recientemente
      target[0][action] = reward

      # Le indicamos al modelo que propague el error hacia atras, pasandole el estado actual, el target, una epoca y verbose igual a 0 para evitar que imprima
      self.model.fit(state, target, epochs=1, verbose=0)

    # Decrementamos
    if self.epsilon > self.epsilon_final:
      self.epsilon *= self.epsilon_decay

## Paso 4: Pre procesado del dataset

### Definir las funciones adicionales

#### Sigmoide

In [14]:
def sigmoid(x):
  return 1 / (1 + math.exp(-x))

#### Función de formato de precios

In [15]:
def stocks_price_format(n):
  if n < 0:
    return "- $ {0:2f}".format(abs(n))
  else:
    return "$ {0:2f}".format(abs(n))

#### Carga del dataset

In [16]:
#import yfinance as yf

def dataset_loader(stock_name):
    # 1. Descarga los datos usando yfinance
    # yf.download() devuelve directamente el DataFrame
    dataset = yf.download(stock_name)

    # 2. Tu lógica original funciona casi igual
    start_date = str(dataset.index[0]).split()[0]
    end_date = str(dataset.index[-1]).split()[0]

    close = dataset['Close']

    return close

# --- Probando la función ---
try:
    aapl_data = dataset_loader("AAPL")
    print("Datos de Apple (AAPL) cargados con éxito:")
    print(aapl_data.head()) # Muestra las primeras 5 filas
except Exception as e:
    print(f"No se pudieron descargar los datos: {e}")

# --- Probando la línea que te daba error ---
try:
    dataset = yf.download("AAPL")
    print("\nPrueba directa con yf.download() exitosa:")
    print(dataset.head())
except Exception as e:
    print(f"\nNo se pudo descargar con yf.download(): {e}")

/tmp/ipython-input-4191731388.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  dataset = yf.download(stock_name)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-4191731388.py:26: FutureWarning: YF.download() has changed argument auto_adjust default to True
  dataset = yf.download("AAPL")
[*********************100%***********************]  1 of 1 completed

Datos de Apple (AAPL) cargados con éxito:
Ticker            AAPL
Date                  
2025-10-10  245.270004
2025-10-13  247.660004
2025-10-14  247.770004
2025-10-15  249.339996
2025-10-16  247.449997

Prueba directa con yf.download() exitosa:
Price            Close        High         Low        Open    Volume
Ticker            AAPL        AAPL        AAPL        AAPL      AAPL
Date                                                                
2025-10-10  245.270004  256.380005  244.000000  254.940002  61999100
2025-10-13  247.660004  249.690002  245.559998  249.380005  38142900
2025-10-14  247.770004  248.850006  244.699997  246.600006  35478000
2025-10-15  249.339996  251.820007  247.470001  249.490005  33893600
2025-10-16  247.449997  249.039993  245.130005  248.250000  39777000


In [17]:
# Esta línea descargará los datos (mostrando el progreso)
dataset = yf.download("AAPL")

# Esta línea mostrará el contenido de la variable
dataset

/tmp/ipython-input-1141241982.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  dataset = yf.download("AAPL")
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2025-10-10,245.270004,256.380005,244.000000,254.940002,61999100
2025-10-13,247.660004,249.690002,245.559998,249.380005,38142900
2025-10-14,247.770004,248.850006,244.699997,246.600006,35478000
2025-10-15,249.339996,251.820007,247.470001,249.490005,33893600
2025-10-16,247.449997,249.039993,245.130005,248.250000,39777000
2025-10-17,252.289993,253.380005,247.270004,248.020004,49147000
2025-10-20,262.239990,264.380005,255.630005,255.889999,90483000
2025-10-21,262.769989,265.290009,261.829987,261.880005,46695900


### State creator

In [ ]:
def state_creator(data, timestep, window_size):

  starting_id = timestep - window_size + 1

  if starting_id >= 0:
    windowed_data = data[starting_id:timestep+1]
  else:
    windowed_data = - starting_id * [data[0]] + list(data[0:timestep+1])

  state = []
  for i in range(window_size - 1):
    state.append(sigmoid(windowed_data[i+1] - windowed_data[i]))

  return np.array([state])

### Cargar una divisa de mercado

In [ ]:
stock_name = "AAPL"
data = dataset_loader(stock_name)

In [ ]:
data.head()

## Paso 5: Entrenar la AI Trader

### Configurar los hyper parámetros

In [ ]:
window_size = 10
episodes = 1000

batch_size = 32
data_samples = len(data) - 1

### Definir el modelo del AI Trader

In [ ]:
trader = AI_Trader(window_size)

In [ ]:
trader.model.summary()

### Bucle de entrenamiento

In [ ]:
for episode in range(1, episodes + 1):

  print("Episodio: {}/{}".format(episode, episodes))

  state = state_creator(data, 0, window_size + 1)

  total_profit = 0
  trader.inventory = []

  for t in tqdm(range(data_samples)):

    action = trader.trade(state)

    next_state = state_creator(data, t+1, window_size + 1)
    reward = 0

    if action == 1: #Compra
      trader.inventory.append(data[t])
      print("AI Trader compró: ", stocks_price_format(data[t]))

    elif action == 2 and len(trader.inventory) > 0: #Vende
      buy_price = trader.inventory.pop(0)

      reward = max(data[t] - buy_price, 0)
      total_profit += data[t] - buy_price
      print("AI Trader vendió: ", stocks_price_format(data[t]), " Beneficio: " + stocks_price_format(data[t] - buy_price) )

    if t == data_samples - 1:
      done = True
    else:
      done = False

    trader.memory.append((state, action, reward, next_state, done))

    state = next_state

    if done:
      print("########################")
      print("BENEFICIO TOTAL: {}".format(total_profit))
      print("########################")

    if len(trader.memory) > batch_size:
      trader.batch_train(batch_size)

  if episode % 10 == 0:
    trader.model.save("ai_trader_{}.h5".format(episode))
